## test_llm_enricher

### Cell 0: Imports & Environment

In [36]:
import os
import json
import textwrap
from datetime import datetime, timezone
from typing import Optional

import snowflake.connector
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, field_validator, model_validator

load_dotenv()

# Verify all required env vars are present before going any further
REQUIRED_VARS = [
    "SNOWFLAKE_ACCOUNT",
    "SNOWFLAKE_USER",
    "SNOWFLAKE_PASSWORD",
    "SNOWFLAKE_ROLE",
    "SNOWFLAKE_WAREHOUSE",
    "SNOWFLAKE_DATABASE",
    "OPENAI_KEY",
]

missing = [v for v in REQUIRED_VARS if not os.getenv(v)]
if missing:
    raise EnvironmentError(f"Missing required env vars: {missing}")

print("✅ All environment variables present")

✅ All environment variables present


### Cell 1a — Snowflake Connection

In [37]:
conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    password=os.environ["SNOWFLAKE_PASSWORD"],
    role=os.environ["SNOWFLAKE_ROLE"],
    warehouse=os.environ["SNOWFLAKE_WAREHOUSE"],
    database=os.environ["SNOWFLAKE_DATABASE"],
)

cur = conn.cursor()

# Confirm we're on the right DB
cur.execute("SELECT CURRENT_ACCOUNT(), CURRENT_DATABASE(), CURRENT_WAREHOUSE()")
row = cur.fetchone()
print(f"✅ Connected — Account: {row[0]} | DB: {row[1]} | WH: {row[2]}")

✅ Connected — Account: FYC39880 | DB: RAW | WH: NYC_JOB_TRACKER_WH


### Cell 1b — Pull 3 Sample Records

In [38]:
SAMPLE_QUERY = """
WITH jsearch_sample AS (
    SELECT
        RAW_PAYLOAD:job_id::STRING        AS job_id,
        SOURCE,
        RAW_PAYLOAD:job_title::STRING     AS job_title,
        RAW_PAYLOAD:job_description::STRING AS description
    FROM RAW.JSEARCH.SRC_POSTINGS
    WHERE RAW_PAYLOAD:job_id::STRING IS NOT NULL
      AND RAW_PAYLOAD:job_id::STRING NOT IN (
          SELECT job_id FROM ENRICHED.PUBLIC.JOB_ENRICHMENT
      )
    LIMIT 1
),
theirstack_sample AS (
    SELECT
        RAW_PAYLOAD:id::STRING            AS job_id,
        SOURCE,
        RAW_PAYLOAD:job_title::STRING     AS job_title,
        RAW_PAYLOAD:description::STRING   AS description
    FROM RAW.THEIRSTACK.SRC_POSTINGS
    WHERE RAW_PAYLOAD:id::STRING IS NOT NULL
      AND RAW_PAYLOAD:id::STRING NOT IN (
          SELECT job_id FROM ENRICHED.PUBLIC.JOB_ENRICHMENT
      )
    LIMIT 1
),
builtin_sample AS (
    SELECT
        RAW_PAYLOAD:identifier:value::STRING AS job_id,
        SOURCE,
        RAW_PAYLOAD:title::STRING            AS job_title,
        RAW_PAYLOAD:description::STRING      AS description
    FROM RAW.BUILTIN.SRC_POSTINGS
    WHERE RAW_PAYLOAD:identifier:value::STRING IS NOT NULL
      AND RAW_PAYLOAD:identifier:value::STRING NOT IN (
          SELECT job_id FROM ENRICHED.PUBLIC.JOB_ENRICHMENT
      )
    LIMIT 1
)
SELECT * FROM jsearch_sample
UNION ALL
SELECT * FROM theirstack_sample
UNION ALL
SELECT * FROM builtin_sample
"""

cur.execute(SAMPLE_QUERY)
sample_rows = cur.fetchall()
col_names = [desc[0].lower() for desc in cur.description]

samples = [dict(zip(col_names, row)) for row in sample_rows]

print(f"✅ Pulled {len(samples)} sample records\n")
for s in samples:
    print(f"  [{s['source']}]  job_id={s['job_id']}")
    print(f"  Title: {s['job_title']}")
    print(f"  Description preview: {str(s['description'])[:120]}...")
    print()

✅ Pulled 3 sample records

  [jsearch:Data Analyst in New York]  job_id=n7ha1uBvkMfQoGz5AAAAAA==
  Title: Data Analyst (New York)
  Description preview: Jobright is a next-generation AI job search platform built to make career navigation faster, smarter, and more personal....

  [theirstack:nyc-data-roles]  job_id=691778902
  Title: Data Analyst
  Description preview: **Location:**
Full Time / 100% Remote (United States)
  
  
**Salary:**
$50,000 - $68,000 per year
  
  
**Job Descripti...

  [builtin]  job_id=9550551
  Title: Investment Quant & Data Analyst
  Description preview: <p><span>As part of the application process, a candidate account is required to log in and view application(s). &nbsp;Pl...



### Cell 2 — OpenAI Connection Ping

In [39]:
client = OpenAI(api_key=os.environ["OPENAI_KEY"])

ping = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Reply with exactly: pong"}
    ],
    max_tokens=10,
    temperature=0,
)

ping_text = ping.choices[0].message.content.strip()
assert "pong" in ping_text.lower(), f"Unexpected ping response: {ping_text}"

print(f"✅ OpenAI connection OK — model=gpt-4o-mini | response='{ping_text}'")
print(f"   Tokens used: {ping.usage.total_tokens}")

✅ OpenAI connection OK — model=gpt-4o-mini | response='pong'
   Tokens used: 13


### Cell 3a — System Prompt

In [40]:
SYSTEM_PROMPT = textwrap.dedent("""
    You are a job posting analyst specializing in data roles.
    Given a job title and description, extract structured metadata.

    Return ONLY a valid JSON object — no preamble, no markdown, no explanation.
    The JSON must conform to this exact schema:

    {
      "inferred_seniority": <"entry" | "mid" | "senior">,
      "is_title_inflated": <true | false>,
      "inflation_reasoning": <string — 1-2 sentences or null if not inflated>,
      "role_archetype": <"data_analyst" | "analytics_engineer" | "data_engineer" | "hybrid" | "software_engineer">,
      "work_focus": <string — short phrase describing primary day-to-day work, max 8 words>,
      "tech_stack_required": <array of lowercase strings — tools explicitly required>,
      "tech_stack_preferred": <array of lowercase strings — tools listed as preferred or nice to have>,
      "paradigms_required": <array of lowercase strings — concepts/practices explicitly required>,
      "paradigms_preferred": <array of lowercase strings — concepts/practices listed as preferred>,
      "degree_requirement": <"none" | "bachelors" | "masters" | "equivalent_ok">,
      "years_required_min": <integer or null>,
      "years_required_max": <integer or null>,
      "salary_min": <integer or null>,
      "salary_max": <integer or null>,
      "confidence_score": <float 0.0–1.0>
    }

    Rules:

    inferred_seniority: base this on actual requirements, NOT the title.
      entry = 0-2 years or no experience required
      mid = 2-5 years
      senior = 5+ years

    is_title_inflated: true if the title implies higher seniority than inferred_seniority.

    role_archetype: the primary operational reality of the role.
      data_analyst = focused on querying, reporting, and business insights
      analytics_engineer = focused on data modeling, transformation, and serving clean data to analysts
      data_engineer = focused on building and maintaining pipelines and infrastructure
      hybrid = genuinely split between two or more data role types with no clear primary
      software_engineer = primarily a software engineering role with minimal data focus

    work_focus: a short free-text phrase (max 8 words) describing what the person in this
      role actually does day to day. Be specific and concrete.
      Good examples:
        "build and maintain ELT pipelines for analytics"
        "create executive dashboards and ad hoc reports"
        "design dimensional models in dbt for analysts"
        "analyze user behavior to inform product decisions"
      Bad examples:
        "data work" (too vague)
        "various data engineering and analytics tasks" (too generic)

    tech_stack_required vs tech_stack_preferred:
      required = explicitly listed under required qualifications or minimum qualifications
      preferred = explicitly listed under preferred, nice to have, or bonus qualifications
      if the posting does not distinguish, put all tools in tech_stack_required
      normalize names: "MS Excel" -> "excel", "Google BigQuery" -> "bigquery", "Apache Airflow" -> "airflow"

    paradigms_required vs paradigms_preferred:
      paradigms are concepts and practices, not tools — e.g. "dimensional modeling", "etl design",
      "data governance", "data warehousing", "pipeline orchestration", "statistical analysis",
      "data modeling", "ml pipelines", "data quality", "data lakes"
      apply the same required vs preferred split as tech stack
      if the posting does not distinguish, put all paradigms in paradigms_required

    degree_requirement:
      none = no degree mentioned or explicitly not required
      bachelors = bachelor's degree explicitly required
      masters = master's degree explicitly required or strongly preferred
      equivalent_ok = degree mentioned but equivalent experience explicitly accepted

    years_required_min / years_required_max:
      parse from required qualifications only — ignore preferred qualifications
      if a single number is given, set min = max = that number
      if no years are mentioned in required qualifications, set both to null

    salary_min / salary_max:
      extract only when explicitly stated in the posting as a number
      always as annual integer (convert hourly if needed: hourly * 2080)
      null if not mentioned

    confidence_score: your confidence that the extraction is accurate given description quality.
      penalize heavily for vague, templated, or very short descriptions
""").strip()

print("System prompt character count:", len(SYSTEM_PROMPT))
print()
print(SYSTEM_PROMPT[:400], "...")

System prompt character count: 4192

You are a job posting analyst specializing in data roles.
Given a job title and description, extract structured metadata.

Return ONLY a valid JSON object — no preamble, no markdown, no explanation.
The JSON must conform to this exact schema:

{
  "inferred_seniority": <"entry" | "mid" | "senior">,
  "is_title_inflated": <true | false>,
  "inflation_reasoning": <string — 1-2 sentences or null if n ...


### Cell 3b — Test Prompt on One Record

In [41]:
test_record = samples[0]

user_message = f"""Job Title: {test_record['job_title']}

Job Description:
{test_record['description']}
"""

print(f"--- Sending to gpt-4o-mini ---")
print(f"Source: {test_record['source']} | job_id: {test_record['job_id']}")
print(f"User message length: {len(user_message)} chars")
print()

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_message},
    ],
    max_tokens=512,
    temperature=0,
    response_format={"type": "json_object"},
)

raw_json_str = response.choices[0].message.content.strip()

print("--- Raw LLM Response ---")
print(raw_json_str)
print()
print(f"Tokens used — prompt: {response.usage.prompt_tokens} | completion: {response.usage.completion_tokens} | total: {response.usage.total_tokens}")

--- Sending to gpt-4o-mini ---
Source: jsearch:Data Analyst in New York | job_id: n7ha1uBvkMfQoGz5AAAAAA==
User message length: 1374 chars

--- Raw LLM Response ---
{
  "inferred_seniority": "entry",
  "is_title_inflated": false,
  "inflation_reasoning": null,
  "role_archetype": "data_analyst",
  "work_focus": "surface data-driven insights for AI agents",
  "tech_stack_required": ["sql", "python", "r"],
  "tech_stack_preferred": [],
  "paradigms_required": ["statistical analysis", "data modeling"],
  "paradigms_preferred": [],
  "degree_requirement": "none",
  "years_required_min": 0,
  "years_required_max": 2,
  "salary_min": null,
  "salary_max": null,
  "confidence_score": 0.9
}

Tokens used — prompt: 1225 | completion: 159 | total: 1384


In [42]:
print(f"=== Sample Record ===")
print(f"Source:  {test_record['source']}")
print(f"job_id:  {test_record['job_id']}")
print(f"Title:   {test_record['job_title']}")
print(f"\n=== Full Description ===")
print(test_record['description'])

=== Sample Record ===
Source:  jsearch:Data Analyst in New York
job_id:  n7ha1uBvkMfQoGz5AAAAAA==
Title:   Data Analyst (New York)

=== Full Description ===
Jobright is a next-generation AI job search platform built to make career navigation faster, smarter, and more personal. They are looking for a Data Analyst to turn raw data into insights that shape how our AI agents learn and improve.

Why Join Us

• Work on AI systems that are actively changing how people find careers
• Your analysis drives real product decisions, not dashboards no one reads

Responsibilities

• Surface data-driven insights that inform how our AI agents are tuned, evaluated, and improved over time
• Build reports and visualizations that make findings clear for both product teams and external stakeholders
• Turn around data requests quickly, translating analytical outputs into accessible narratives

Qualifications

Required

• Recent grad or early-career professional with 0 to 2 years of experience in data analysi

### Cell 4 — Pydantic Validation

In [46]:
from typing import List, Literal, Optional

class JobEnrichmentSchema(BaseModel):
    inferred_seniority:   Literal["entry", "mid", "senior"]
    is_title_inflated:    bool
    inflation_reasoning:  Optional[str] = None
    role_archetype:       Literal["data_analyst", "analytics_engineer", "data_engineer", "hybrid", "software_engineer"]
    work_focus:           str
    tech_stack_required:  List[str]
    tech_stack_preferred: List[str]
    paradigms_required:   List[str]
    paradigms_preferred:  List[str]
    degree_requirement:   Literal["none", "bachelors", "masters", "equivalent_ok"]
    years_required_min:   Optional[int] = None
    years_required_max:   Optional[int] = None
    salary_min:           Optional[int] = None
    salary_max:           Optional[int] = None
    confidence_score:     float

    @field_validator("work_focus")
    @classmethod
    def validate_work_focus(cls, v: str) -> str:
        words = v.strip().split()
        if len(words) > 10:
            raise ValueError(f"work_focus too long ({len(words)} words), max 8 words: '{v}'")
        return v.strip().lower()

    @field_validator("confidence_score")
    @classmethod
    def validate_confidence(cls, v: float) -> float:
        if not (0.0 <= v <= 1.0):
            raise ValueError(f"confidence_score must be 0.0–1.0, got {v}")
        return v

    @field_validator("tech_stack_required", "tech_stack_preferred")
    @classmethod
    def normalize_stack(cls, v: List[str]) -> List[str]:
        return [item.lower().strip() for item in v]

    @field_validator("paradigms_required", "paradigms_preferred")
    @classmethod
    def normalize_paradigms(cls, v: List[str]) -> List[str]:
        return [item.lower().strip() for item in v]

    @model_validator(mode="after")
    def validate_years_range(self) -> "JobEnrichmentSchema":
        lo, hi = self.years_required_min, self.years_required_max
        if lo is not None and hi is not None and lo > hi:
            raise ValueError(f"years_required_min ({lo}) > years_required_max ({hi})")
        return self

print("✅ JobEnrichmentSchema defined")

parsed_data = json.loads(raw_json_str)
enrichment = JobEnrichmentSchema(**parsed_data)

print("✅ Pydantic validation passed\n")
print("Validated fields:")
for field, value in enrichment.model_dump().items():
    print(f"  {field:25s}: {value}")

✅ JobEnrichmentSchema defined
✅ Pydantic validation passed

Validated fields:
  inferred_seniority       : entry
  is_title_inflated        : False
  inflation_reasoning      : None
  role_archetype           : data_analyst
  work_focus               : surface data-driven insights for ai agents
  tech_stack_required      : ['sql', 'python', 'r']
  tech_stack_preferred     : []
  paradigms_required       : ['statistical analysis', 'data modeling']
  paradigms_preferred      : []
  degree_requirement       : none
  years_required_min       : 0
  years_required_max       : 2
  salary_min               : None
  salary_max               : None
  confidence_score         : 0.9


### Cell 5 — Preview Insert Row

In [47]:
MODEL_VERSION = "gpt-4o-mini"

insert_row = {
    "job_id":               test_record["job_id"],
    "source":               test_record["source"],
    "inferred_seniority":   enrichment.inferred_seniority,
    "is_title_inflated":    enrichment.is_title_inflated,
    "inflation_reasoning":  enrichment.inflation_reasoning,
    "role_archetype":       enrichment.role_archetype,
    "work_focus":           enrichment.work_focus,
    "tech_stack_required":  json.dumps(enrichment.tech_stack_required),
    "tech_stack_preferred": json.dumps(enrichment.tech_stack_preferred),
    "paradigms_required":   json.dumps(enrichment.paradigms_required),
    "paradigms_preferred":  json.dumps(enrichment.paradigms_preferred),
    "degree_requirement":   enrichment.degree_requirement,
    "years_required_min":   enrichment.years_required_min,
    "years_required_max":   enrichment.years_required_max,
    "salary_min":           enrichment.salary_min,
    "salary_max":           enrichment.salary_max,
    "confidence_score":     enrichment.confidence_score,
    "enriched_at":          datetime.now(timezone.utc).isoformat(),
    "model_version":        MODEL_VERSION,
}

print("--- Preview: Snowflake Insert Row ---")
print(f"  Table: enriched.public.job_enrichment")
print()
for col, val in insert_row.items():
    print(f"  {col:25s}: {val}")

print()
print("✅ Insert row looks valid — ready to proceed to full batch.")

--- Preview: Snowflake Insert Row ---
  Table: enriched.public.job_enrichment

  job_id                   : n7ha1uBvkMfQoGz5AAAAAA==
  source                   : jsearch:Data Analyst in New York
  inferred_seniority       : entry
  is_title_inflated        : False
  inflation_reasoning      : None
  role_archetype           : data_analyst
  work_focus               : surface data-driven insights for ai agents
  tech_stack_required      : ["sql", "python", "r"]
  tech_stack_preferred     : []
  paradigms_required       : ["statistical analysis", "data modeling"]
  paradigms_preferred      : []
  degree_requirement       : none
  years_required_min       : 0
  years_required_max       : 2
  salary_min               : None
  salary_max               : None
  confidence_score         : 0.9
  enriched_at              : 2026-06-03T19:58:23.302741+00:00
  model_version            : gpt-4o-mini

✅ Insert row looks valid — ready to proceed to full batch.


### Cell 6a — Pull All Unenriched Jobs

In [50]:
FULL_BATCH_QUERY = """
WITH jsearch AS (
    SELECT
        RAW_PAYLOAD:job_id::STRING          AS job_id,
        SOURCE,
        RAW_PAYLOAD:job_title::STRING       AS job_title,
        RAW_PAYLOAD:job_description::STRING AS description
    FROM RAW.JSEARCH.SRC_POSTINGS
    WHERE RAW_PAYLOAD:job_id::STRING IS NOT NULL
      AND RAW_PAYLOAD:job_id::STRING NOT IN (
          SELECT job_id FROM ENRICHED.PUBLIC.JOB_ENRICHMENT
      )
),
theirstack AS (
    SELECT
        RAW_PAYLOAD:id::STRING              AS job_id,
        SOURCE,
        RAW_PAYLOAD:job_title::STRING       AS job_title,
        RAW_PAYLOAD:description::STRING     AS description
    FROM RAW.THEIRSTACK.SRC_POSTINGS
    WHERE RAW_PAYLOAD:id::STRING IS NOT NULL
      AND RAW_PAYLOAD:id::STRING NOT IN (
          SELECT job_id FROM ENRICHED.PUBLIC.JOB_ENRICHMENT
      )
),
builtin AS (
    SELECT
        RAW_PAYLOAD:identifier:value::STRING AS job_id,
        SOURCE,
        RAW_PAYLOAD:title::STRING            AS job_title,
        RAW_PAYLOAD:description::STRING      AS description
    FROM RAW.BUILTIN.SRC_POSTINGS
    WHERE RAW_PAYLOAD:identifier:value::STRING IS NOT NULL
      AND RAW_PAYLOAD:identifier:value::STRING NOT IN (
          SELECT job_id FROM ENRICHED.PUBLIC.JOB_ENRICHMENT
      )
)
SELECT * FROM jsearch
UNION ALL
SELECT * FROM theirstack
UNION ALL
SELECT * FROM builtin
ORDER BY source, job_id
"""

cur.execute(FULL_BATCH_QUERY)
all_rows = cur.fetchall()
col_names = [desc[0].lower() for desc in cur.description]
all_jobs = [dict(zip(col_names, row)) for row in all_rows]

print(f"📋 Total unenriched jobs: {len(all_jobs)}")

from collections import Counter
source_counts = Counter(j["source"] for j in all_jobs)
for src, count in sorted(source_counts.items()):
    print(f"   {src}: {count}")

📋 Total unenriched jobs: 52
   builtin: 7
   jsearch:Analytics Engineer in New York: 20
   jsearch:Data Analyst in New York: 21
   theirstack: 3
   theirstack:nyc-data-roles: 1


### Cell 6b — Run Full Batch (No Writes)

In [53]:
enrichment_results = []
failed_jobs = []

for i, job in enumerate(all_jobs):
    job_id = job["job_id"]
    source = job["source"]

    if not job.get("description"):
        print(f"[{i+1}/{len(all_jobs)}] SKIP (no description) — {source} | {job_id}")
        failed_jobs.append({"job_id": job_id, "source": source, "error": "empty description"})
        continue

    user_msg = f"Job Title: {job['job_title']}\n\nJob Description:\n{job['description']}"

    try:
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": user_msg},
            ],
            max_tokens=512,
            temperature=0,
            response_format={"type": "json_object"},
        )

        raw = resp.choices[0].message.content.strip()
        parsed = json.loads(raw)
        validated = JobEnrichmentSchema(**parsed)

        row = {
            "job_id":               job_id,
            "source":               source,
            "inferred_seniority":   validated.inferred_seniority,
            "is_title_inflated":    validated.is_title_inflated,
            "inflation_reasoning":  validated.inflation_reasoning,
            "role_archetype":       validated.role_archetype,
            "work_focus":           validated.work_focus,
            "tech_stack_required":  json.dumps(validated.tech_stack_required),
            "tech_stack_preferred": json.dumps(validated.tech_stack_preferred),
            "paradigms_required":   json.dumps(validated.paradigms_required),
            "paradigms_preferred":  json.dumps(validated.paradigms_preferred),
            "degree_requirement":   validated.degree_requirement,
            "years_required_min":   validated.years_required_min,
            "years_required_max":   validated.years_required_max,
            "salary_min":           validated.salary_min,
            "salary_max":           validated.salary_max,
            "confidence_score":     validated.confidence_score,
            "enriched_at":          datetime.now(timezone.utc).isoformat(),
            "model_version":        MODEL_VERSION,
        }
        enrichment_results.append(row)
        print(f"[{i+1}/{len(all_jobs)}] OK — {source} | {job_id} | archetype={validated.role_archetype} | seniority={validated.inferred_seniority} | inflated={validated.is_title_inflated} | confidence={validated.confidence_score:.2f}")

    except Exception as e:
        print(f"[{i+1}/{len(all_jobs)}] FAIL — {source} | {job_id} | error: {e}")
        failed_jobs.append({"job_id": job_id, "source": source, "error": str(e)})

print()
print(f"✅ Enrichment complete — {len(enrichment_results)} succeeded, {len(failed_jobs)} failed")

[1/52] OK — builtin | 9526440 | archetype=data_analyst | seniority=mid | inflated=False | confidence=0.90
[2/52] OK — builtin | 9527826 | archetype=data_analyst | seniority=entry | inflated=True | confidence=0.85
[3/52] OK — builtin | 9536284 | archetype=data_analyst | seniority=mid | inflated=False | confidence=0.90
[4/52] OK — builtin | 9547634 | archetype=data_analyst | seniority=mid | inflated=False | confidence=0.90
[5/52] OK — builtin | 9549753 | archetype=data_analyst | seniority=mid | inflated=False | confidence=0.90
[6/52] OK — builtin | 9550265 | archetype=data_analyst | seniority=mid | inflated=False | confidence=0.85
[7/52] OK — builtin | 9550551 | archetype=data_analyst | seniority=entry | inflated=False | confidence=0.85
[8/52] OK — jsearch:Analytics Engineer in New York | 2CxwkUVr5mty-fZ2AAAAAA== | archetype=software_engineer | seniority=senior | inflated=False | confidence=0.90
[9/52] OK — jsearch:Analytics Engineer in New York | 3bN4M4270UDIg2dtAAAAAA== | archetype=sof

### Cell 6c — Preview Batch Results

In [54]:
import pandas as pd

df = pd.DataFrame(enrichment_results)

print(f"=== Batch Results Preview ({len(df)} rows) ===\n")

print("--- Role Archetype Distribution ---")
print(df["role_archetype"].value_counts().to_string())
print()

print("--- Seniority Distribution ---")
print(df["inferred_seniority"].value_counts().to_string())
print()

print("--- Title Inflation Rate ---")
print(df["is_title_inflated"].value_counts().to_string())
print(f"Inflation rate: {df['is_title_inflated'].mean():.1%}")
print()

print("--- Degree Requirement Distribution ---")
print(df["degree_requirement"].value_counts().to_string())
print()

print("--- Salary Coverage ---")
has_salary = df["salary_min"].notna().sum()
print(f"Postings with salary data: {has_salary}/{len(df)} ({has_salary/len(df):.1%})")
if has_salary > 0:
    print(f"Salary range: ${df['salary_min'].min():,.0f} – ${df['salary_max'].max():,.0f}")
print()

print("--- Confidence Score Stats ---")
print(df["confidence_score"].describe().round(3).to_string())
print()

print("--- Source Breakdown ---")
print(df["source"].value_counts().to_string())
print()

print("--- Work Focus Sample (first 10) ---")
for i, row in df.head(10).iterrows():
    print(f"  [{row['role_archetype']:20s}] {row['work_focus']}")
print()

print("--- Top Required Tools ---")
from collections import Counter
required_tools = Counter()
for stack in df["tech_stack_required"]:
    if stack:
        required_tools.update(json.loads(stack) if isinstance(stack, str) else stack)
for tool, count in required_tools.most_common(15):
    print(f"  {tool:25s}: {count}")
print()

print("--- Top Required Paradigms ---")
required_paradigms = Counter()
for p in df["paradigms_required"]:
    if p:
        required_paradigms.update(json.loads(p) if isinstance(p, str) else p)
for paradigm, count in required_paradigms.most_common(15):
    print(f"  {paradigm:25s}: {count}")
print()

print("--- First 10 Rows ---")
display_cols = [
    "job_id", "source", "role_archetype", "inferred_seniority",
    "is_title_inflated", "degree_requirement", "confidence_score",
    "years_required_min", "years_required_max"
]
pd.set_option("display.max_colwidth", 30)
print(df[display_cols].head(10).to_string(index=False))

if failed_jobs:
    print("\n--- Failed Jobs ---")
    for f in failed_jobs:
        print(f"  {f['source']} | {f['job_id']} | {f['error']}")

print()
print("⚠️  DRY RUN COMPLETE — No rows written to Snowflake.")
print(f"   When ready to write: pass `enrichment_results` to llm_enricher.py's write_to_snowflake()")

=== Batch Results Preview (52 rows) ===

--- Role Archetype Distribution ---
role_archetype
data_analyst          28
data_engineer          9
hybrid                 8
software_engineer      4
analytics_engineer     3

--- Seniority Distribution ---
inferred_seniority
mid       26
senior    14
entry     12

--- Title Inflation Rate ---
is_title_inflated
False    39
True     13
Inflation rate: 25.0%

--- Degree Requirement Distribution ---
degree_requirement
none             19
bachelors        18
equivalent_ok     9
masters           6

--- Salary Coverage ---
Postings with salary data: 32/52 (61.5%)
Salary range: $50,000 – $260,100

--- Confidence Score Stats ---
count    52.000
mean      0.866
std       0.073
min       0.500
25%       0.850
50%       0.900
75%       0.900
max       0.900

--- Source Breakdown ---
source
jsearch:Data Analyst in New York          21
jsearch:Analytics Engineer in New York    20
builtin                                    7
theirstack                      

### Cell 6d - LLM Response Explorer

In [59]:
# -----------------------------------------------------------------------
# EXPLORER — change this index to inspect any record
# -----------------------------------------------------------------------
INSPECT_INDEX = 4  # <-- change this to any number 0–51

# stash
# 11 - Data Analytics Engineer, YouTube

# -----------------------------------------------------------------------
record = enrichment_results[INSPECT_INDEX]
job = all_jobs[INSPECT_INDEX]

print(f"{'='*60}")
print(f"Record {INSPECT_INDEX + 1} of {len(enrichment_results)}")
print(f"{'='*60}")
print(f"Source:     {record['source']}")
print(f"Title:      {job['job_title']}")
print(f"job_id:     {record['job_id']}")
print(f"\n--- Full Description ---")
print(job['description'])
print(f"\n--- LLM Output ---")
print(f"  role_archetype      : {record['role_archetype']}")
print(f"  work_focus          : {record['work_focus']}")
print(f"  inferred_seniority  : {record['inferred_seniority']}")
print(f"  is_title_inflated   : {record['is_title_inflated']}")
print(f"  inflation_reasoning : {record['inflation_reasoning']}")
print(f"  degree_requirement  : {record['degree_requirement']}")
print(f"  years_required_min  : {record['years_required_min']}")
print(f"  years_required_max  : {record['years_required_max']}")
print(f"  salary_min          : {record['salary_min']}")
print(f"  salary_max          : {record['salary_max']}")
print(f"  tech_stack_required : {record['tech_stack_required']}")
print(f"  tech_stack_preferred: {record['tech_stack_preferred']}")
print(f"  paradigms_required  : {record['paradigms_required']}")
print(f"  paradigms_preferred : {record['paradigms_preferred']}")
print(f"  confidence_score    : {record['confidence_score']}")

Record 5 of 52
Source:     builtin
Title:      Data Analyst and Reporting Specialist
job_id:     9549753

--- Full Description ---
<b>Job Summary &amp; Responsibilities</b><p><span>IEM is searching for a <strong>Full Time- Data Analyst and Reporting Specialist</strong> to provide data analytics and reporting support to IEM’s Disaster Response and Recovery management and other IEM staff.&nbsp;The Data Analyst and Reporting Specialist is responsible for transforming complex, multi‑source data into accurate, actionable insights that support operational, financial, and executive decision‑making across the organization. This role designs and maintains scalable Power BI data models, dashboards, and reporting solutions while ensuring strong data quality, governance, security, and performance within the BI environment. Working collaboratively with leadership, technical teams, and client stakeholders, the specialist gathers and interprets business requirements, streamlines data processes, and d

### Cell 7 — Cleanup

In [ ]:
# cur.close()
# conn.close()
# print("✅ Snowflake connection closed")

### Test Cell A - Insert one row

In [62]:
test_conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    password=os.environ["SNOWFLAKE_PASSWORD"],
    role=os.environ["SNOWFLAKE_ROLE"],
    warehouse=os.environ["SNOWFLAKE_WAREHOUSE"],
    database=os.environ["SNOWFLAKE_DATABASE"],
)
test_cur = test_conn.cursor()

INSERT_QUERY = """
INSERT INTO enriched.public.job_enrichment (
    job_id, source, inferred_seniority, is_title_inflated, inflation_reasoning,
    role_archetype, work_focus, tech_stack_required, tech_stack_preferred,
    paradigms_required, paradigms_preferred, degree_requirement,
    years_required_min, years_required_max, salary_min, salary_max,
    confidence_score, enriched_at, model_version
)
SELECT
    $1, $2, $3, $4, $5, $6, $7,
    PARSE_JSON($8), PARSE_JSON($9), PARSE_JSON($10), PARSE_JSON($11),
    $12, $13, $14, $15, $16, $17, $18::TIMESTAMP_TZ, $19
FROM VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

test_cur.execute(INSERT_QUERY, (
    insert_row["job_id"], insert_row["source"], insert_row["inferred_seniority"],
    insert_row["is_title_inflated"], insert_row["inflation_reasoning"],
    insert_row["role_archetype"], insert_row["work_focus"],
    insert_row["tech_stack_required"], insert_row["tech_stack_preferred"],
    insert_row["paradigms_required"], insert_row["paradigms_preferred"],
    insert_row["degree_requirement"], insert_row["years_required_min"],
    insert_row["years_required_max"], insert_row["salary_min"], insert_row["salary_max"],
    insert_row["confidence_score"], insert_row["enriched_at"], insert_row["model_version"],
))

print("✅ Row inserted — go check Snowflake UI now")
print(f"   job_id: {insert_row['job_id']}")

✅ Row inserted — go check Snowflake UI now
   job_id: n7ha1uBvkMfQoGz5AAAAAA==


### Test Cell B — Verify it landed

In [63]:
test_cur.execute("""
    SELECT job_id, source, role_archetype, inferred_seniority,
           tech_stack_required, paradigms_required, enriched_at
    FROM enriched.public.job_enrichment
    WHERE job_id = %s
""", (insert_row["job_id"],))

row = test_cur.fetchone()
print(f"✅ Row verified in Snowflake:")
print(f"  job_id        : {row[0]}")
print(f"  source        : {row[1]}")
print(f"  role_archetype: {row[2]}")
print(f"  seniority     : {row[3]}")
print(f"  tech_required : {row[4]}")
print(f"  paradigms     : {row[5]}")
print(f"  enriched_at   : {row[6]}")

✅ Row verified in Snowflake:
  job_id        : n7ha1uBvkMfQoGz5AAAAAA==
  source        : jsearch:Data Analyst in New York
  role_archetype: data_analyst
  seniority     : entry
  tech_required : [
  "sql",
  "python",
  "r"
]
  paradigms     : [
  "statistical analysis",
  "data modeling"
]
  enriched_at   : 2026-06-03 19:58:23.302741+00:00


### Test Cell C — Delete the test row

In [64]:
test_cur.execute("""
    DELETE FROM enriched.public.job_enrichment
    WHERE job_id = %s
""", (insert_row["job_id"],))

print(f"✅ Test row deleted — table is clean")

test_cur.close()
test_conn.close()

✅ Test row deleted — table is clean
